In [1]:

 # Cellule 1 - Imports et configuration
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import joblib
import os
import pandas as pd
import numpy as np
import logging

# Configuration du logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Chemins des modèles
MODEL_DIR = "models"
MODEL_PATH = os.path.join(MODEL_DIR, "isolation_forest_v1.joblib")
SCALER_PATH = os.path.join(MODEL_DIR, "scaler_v1.joblib")

# Création du dossier models si inexistant
os.makedirs(MODEL_DIR, exist_ok=True)


In [35]:
import sys
!"{sys.executable}" -m pip install xgboost --upgrade


  Using cached xgboost-3.0.5-py3-none-win_amd64.whl.metadata (2.1 kB)
Using cached xgboost-3.0.5-py3-none-win_amd64.whl (56.8 MB)


In [36]:
from xgboost import XGBClassifier
print("XGBoost est installé et prêt !")


XGBoost est installé et prêt !


In [50]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import re
from datetime import datetime, timedelta
import os

# Create models directory if it doesn't exist
os.makedirs('models', exist_ok=True)

class XGBoostTrainer:
    def __init__(self):
        self.model = xgb.XGBClassifier(
            n_estimators=200,
            max_depth=8,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            eval_metric='logloss',
            use_label_encoder=False
        )
        self.scaler = StandardScaler()
        self.feature_importance = None
    
    def load_and_preprocess_data(self, file_path):
        """Charge et préprocesse les données avec TOUTES les features attendues"""
        print("Chargement des données...")
        df = pd.read_csv(file_path, encoding='utf-8')
        
        # Conversion des dates
        df['creDtTm'] = pd.to_datetime(df['creDtTm'])
        
        # === AJOUTER TOUTES LES FEATURES ATTENDUES ===
        
        # Features de base
        df['intrbk_sttlm_amt_log'] = np.log1p(df['intrbk_sttlm_amt'].clip(lower=0) + 1e-6)
        df['is_international'] = (df['dbtrCtry'] != df['cdtrCtry']).astype(int)
        df['same_country'] = (df['dbtrCtry'] == df['cdtrCtry']).astype(int)
        df['same_bank'] = (df['dbtrAgtFinInstnId'] == df['cdtrAgtFinInstnId']).astype(int)
        
        # Features temporelles avancées
        df['hour_of_day'] = df['creDtTm'].dt.hour
        df['day_of_week'] = df['creDtTm'].dt.dayofweek
        
        # Encodage cyclique
        df['hour_of_day_sin'] = np.sin(2 * np.pi * df['hour_of_day'] / 24)
        df['hour_of_day_cos'] = np.cos(2 * np.pi * df['hour_of_day'] / 24)
        df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
        df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
        df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
        df['is_night'] = ((df['hour_of_day'] >= 22) | (df['hour_of_day'] <= 6)).astype(int)
        
        # Features de pattern
        df['name_similarity'] = df.apply(
            lambda row: self._calculate_name_similarity(row['dbtrNm'], row['cdtrNm']), axis=1
        )
        df['amount_roundness'] = df.apply(
            lambda row: self._calculate_roundness(row['intrbk_sttlm_amt']), axis=1
        )
        df['account_pattern_risk'] = df.apply(
            lambda row: self._detect_suspicious_pattern(row['dbtrAcctId']), axis=1
        )
        df['debtor_creditor_same_sector'] = df.apply(
            lambda row: self._detect_same_sector(row['dbtrNm'], row['cdtrNm']), axis=1
        )
        
        # Features statistiques
        df['amount_zscore'] = (df['intrbk_sttlm_amt'] - df['intrbk_sttlm_amt'].mean()) / df['intrbk_sttlm_amt'].std()
        
        # Features combinatoires
        df['cross_border_and_urgent'] = (
            (df['is_international'] == 1) & 
            ((df['InstrPrty'] == 'HIGH') | (df['SvcLvl'] == 'URGP'))
        ).astype(int)
        
        df['high_risk_transaction'] = (
            (df['is_international'] == 1) & 
            (df['intrbk_sttlm_amt'] > df['intrbk_sttlm_amt'].quantile(0.9))
        ).astype(int)
        
        # Flags de montant
        high_amount_threshold = df['intrbk_sttlm_amt'].quantile(0.99) if len(df) > 10 else 500000
        low_amount_threshold = df['intrbk_sttlm_amt'].quantile(0.01) if len(df) > 10 else 10
        
        df['high_amount_flag'] = (df['intrbk_sttlm_amt'] > high_amount_threshold).astype(int)
        df['extreme_amount'] = (
            (df['intrbk_sttlm_amt'] > high_amount_threshold) |
            (df['intrbk_sttlm_amt'] < low_amount_threshold)
        ).astype(int)
        
        # Features de distance de Mahalanobis (simplifiée pour l'entraînement)
        numeric_cols = ['intrbk_sttlm_amt', 'distance_km', 'amount_zscore']
        numeric_data = df[numeric_cols].fillna(0)
        
        if len(df) > 5:
            try:
                cov_matrix = np.cov(numeric_data.T)
                mean_vec = numeric_data.mean().values
                inv_cov = np.linalg.pinv(cov_matrix)
                
                df['mahalanobis_distance'] = df.apply(
                    lambda row: self._calculate_mahalanobis(
                        row[numeric_cols].fillna(0).values, 
                        mean_vec, 
                        inv_cov
                    ), axis=1
                )
            except:
                df['mahalanobis_distance'] = 0
        else:
            df['mahalanobis_distance'] = 0
        
        # Features comportementales (simulées pour l'entraînement)
        df['debtor_tx_count_1h'] = np.random.randint(0, 5, len(df))  # Simulation
        df['debtor_tx_count_24h'] = np.random.randint(0, 20, len(df))
        df['creditor_tx_count_1h'] = np.random.randint(0, 5, len(df))
        df['creditor_tx_count_24h'] = np.random.randint(0, 20, len(df))
        df['debtor_amount_std_24h'] = np.random.uniform(0, 1000, len(df))
        df['creditor_amount_std_24h'] = np.random.uniform(0, 1000, len(df))
        
        return df
    
    def _calculate_name_similarity(self, name1, name2):
        """Calcule la similarité entre deux noms"""
        name1 = str(name1).lower().replace(' ', '')
        name2 = str(name2).lower().replace(' ', '')
        if not name1 or not name2:
            return 0
        return sum(1 for a, b in zip(name1, name2) if a == b) / max(len(name1), len(name2))
    
    def _calculate_roundness(self, amount):
        """Calcule le roundness d'un montant"""
        if amount == 0:
            return 0
        rounded = round(amount / 1000) * 1000
        return 1 - min(abs(amount - rounded) / amount, 1)
    
    def _detect_suspicious_pattern(self, account_number):
        """Détecte les patterns suspects"""
        an = str(account_number)
        if len(an) < 5:
            return 0
        
        # Séquence répétitive
        if re.match(r'(\d)\1{3,}', an):
            return 1
        
        # Séquence séquentielle
        if any(an[i:i+4] in ['1234', '4321', '0000', '9999'] for i in range(len(an)-3)):
            return 1
        
        return 0
    
    def _detect_same_sector(self, name1, name2):
        """Détecte si débiteur et créditeur sont dans le même secteur"""
        def detect_sector(name):
            name = str(name).lower()
            if any(word in name for word in ['industr', 'atlas', 'matières', 'premières']):
                return 'INDUSTRIEL'
            elif any(word in name for word in ['distrib', 'ventes', 'commerce']):
                return 'DISTRIBUTION'
            elif any(word in name for word in ['textile', 'habillement']):
                return 'TEXTILE'
            elif any(word in name for word in ['agricol', 'farm', 'produit']):
                return 'AGRICOLE'
            elif any(word in name for word in ['touris', 'travel', 'hotel', 'voyage']):
                return 'TOURISME'
            else:
                return 'SERVICES'
        
        sector1 = detect_sector(name1)
        sector2 = detect_sector(name2)
        return 1 if sector1 == sector2 else 0
    
    def _calculate_mahalanobis(self, x, mean, inv_cov):
        """Calcule la distance de Mahalanobis"""
        try:
            diff = x - mean
            return np.sqrt(diff.dot(inv_cov).dot(diff))
        except:
            return 0
    
    def prepare_features(self, df):
        """Prépare les features pour l'entraînement avec TOUTES les features attendues"""
        # Utiliser EXACTEMENT les mêmes features que dans FraudModel
        features = [
            # 1️⃣ Features financières
            'intrbk_sttlm_amt', 
            'intrbk_sttlm_amt_log', 
            'amount_roundness',
            'high_amount_flag',
            'extreme_amount',
            
            # 2️⃣ Features géographiques / banque
            'is_international',
            'same_country',
            'same_bank',
            'distance_km',
            
            # 3️⃣ Features textuelles / patterns
            'name_similarity',
            'account_pattern_risk',
            'debtor_creditor_same_sector',
            
            # 4️⃣ Features temporelles
            'hour_of_day_sin',
            'hour_of_day_cos',
            'day_of_week_sin',
            'day_of_week_cos',
            'is_weekend',
            'is_night',
            
            # 5️⃣ Features combinatoires / dérivées
            'cross_border_and_urgent',
            'high_risk_transaction',
            'amount_zscore',
            'mahalanobis_distance'
        ]
        
        # S'assurer que toutes les features sont présentes
        available_features = [f for f in features if f in df.columns]
        missing_features = [f for f in features if f not in df.columns]
        
        if missing_features:
            print(f"ATTENTION: Features manquantes: {missing_features}")
            # Ajouter les features manquantes avec des valeurs par défaut
            for feature in missing_features:
                df[feature] = 0
        
        X = df[features].copy()
        y = df['is_fraud']
        
        return X, y
    
    def train(self, X, y):
        """Entraîne le modèle"""
        print("Division train/test...")
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        
        print("Scaling des features...")
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)
        
        print("Entraînement du modèle XGBoost...")
        self.model.fit(X_train_scaled, y_train)
        
        # Prédictions
        y_pred = self.model.predict(X_test_scaled)
        y_pred_proba = self.model.predict_proba(X_test_scaled)[:, 1]
        
        # Évaluation
        print("\n=== RAPPORT DE CLASSIFICATION ===")
        print(classification_report(y_test, y_pred))
        
        print("\n=== MATRICE DE CONFUSION ===")
        print(confusion_matrix(y_test, y_pred))
        
        print(f"\n=== AUC-ROC: {roc_auc_score(y_test, y_pred_proba):.4f} ===")
        
        # Feature importance
        self.feature_importance = pd.DataFrame({
            'feature': X.columns,
            'importance': self.model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        return X_test_scaled, y_test, y_pred
    
    def plot_feature_importance(self):
        """Affiche l'importance des features"""
        plt.figure(figsize=(12, 10))
        sns.barplot(x='importance', y='feature', data=self.feature_importance.head(20))
        plt.title('Importance des Features - XGBoost')
        plt.tight_layout()
        plt.savefig('feature_importance.png')
        plt.show()
    
    def save_model(self, model_path='models/xgboost_fraud_model.joblib', 
                  scaler_path='models/scaler.joblib'):
        """Sauvegarde le modèle et le scaler"""
        joblib.dump(self.model, model_path)
        joblib.dump(self.scaler, scaler_path)
        print(f"Modèle sauvegardé: {model_path}")
        print(f"Scaler sauvegardé: {scaler_path}")

# Entraînement du modèle
def main():
    # Initialisation
    trainer = XGBoostTrainer()
    
    # Chargement des données avec TOUTES les features
    df = trainer.load_and_preprocess_data('maroc_iso20022_transactions_v2.csv')
    
    # Préparation des features
    X, y = trainer.prepare_features(df)
    
    # Vérification des features
    print("Features disponibles:", X.columns.tolist())
    print("Shape des données:", X.shape)
    print("Nombre de fraudes:", y.sum())
    print("Pourcentage de fraudes:", f"{y.mean()*100:.2f}%")
    
    # Entraînement
    X_test, y_test, y_pred = trainer.train(X, y)
    
    # Visualisation
    trainer.plot_feature_importance()
    
    # Sauvegarde
    trainer.save_model()
    
    print("\n=== ENTRAÎNEMENT TERMINÉ ===")

if __name__ == "__main__":
    main()

Chargement des données...


C:\Users\Bellamine Kenza\AppData\Local\Temp\ipykernel_31904\2215221102.py:110: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  row[numeric_cols].fillna(0).values,
C:\Users\Bellamine Kenza\AppData\Local\Temp\ipykernel_31904\2215221102.py:110: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  row[numeric_cols].fillna(0).values,
C:\Users\Bellamine Kenza\AppData\Local\Temp\ipykernel_31904\2215221102.py:110: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=Fal

Features disponibles: ['intrbk_sttlm_amt', 'intrbk_sttlm_amt_log', 'amount_roundness', 'high_amount_flag', 'extreme_amount', 'is_international', 'same_country', 'same_bank', 'distance_km', 'name_similarity', 'account_pattern_risk', 'debtor_creditor_same_sector', 'hour_of_day_sin', 'hour_of_day_cos', 'day_of_week_sin', 'day_of_week_cos', 'is_weekend', 'is_night', 'cross_border_and_urgent', 'high_risk_transaction', 'amount_zscore', 'mahalanobis_distance']
Shape des données: (15000, 22)
Nombre de fraudes: 2676
Pourcentage de fraudes: 17.84%
Division train/test...
Scaling des features...
Entraînement du modèle XGBoost...


c:\Users\Bellamine Kenza\Desktop\fraud_new\venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:32:52] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



=== RAPPORT DE CLASSIFICATION ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2465
           1       1.00      1.00      1.00       535

    accuracy                           1.00      3000
   macro avg       1.00      1.00      1.00      3000
weighted avg       1.00      1.00      1.00      3000


=== MATRICE DE CONFUSION ===
[[2464    1]
 [   0  535]]

=== AUC-ROC: 1.0000 ===
Modèle sauvegardé: models/xgboost_fraud_model.joblib
Scaler sauvegardé: models/scaler.joblib

=== ENTRAÎNEMENT TERMINÉ ===


C:\Users\Bellamine Kenza\AppData\Local\Temp\ipykernel_31904\2215221102.py:284: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [48]:
# Ajoutez ces imports
import re
from datetime import datetime, timedelta

def main():
    # Initialisation
    trainer = XGBoostTrainer()
    
    # Chargement des données avec TOUTES les features
    df = trainer.load_and_preprocess_data('maroc_iso20022_transactions.csv')
    
    # Préparation des features
    X, y = trainer.prepare_features(df)
    
    # Vérification des features
    print("Features disponibles:", X.columns.tolist())
    print("Shape des données:", X.shape)
    
    # Entraînement
    X_test, y_test, y_pred = trainer.train(X, y)
    
    # Visualisation
    trainer.plot_feature_importance()
    
    # Sauvegarde
    trainer.save_model(
        model_path='models/xgboost_fraud_model.joblib',
        scaler_path='models/scaler.joblib'
    )
    
    print("\n=== ENTRAÎNEMENT TERMINÉ ===")